In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

print("Libraries loaded successfully!")

In [2]:
df = pd.read_csv("products.csv")

In [3]:
print(f"total products loaded: {len(df)}")
df.head()

total products loaded: 12


,name,description,sku,category
0,Samsung 55 Inch QLED TV,4K Ultra HD Smart TV with Quantum Dot technolo...,SAM-TV-55Q,Electronics
1,Sony WH-1000XM5 Headphones,Wireless noise cancelling headphones with 30 h...,SNY-HP-XM5,Electronics
2,Nike Air Max 270 Running Shoes,Lightweight athletic shoes with Air Max cushio...,NKE-SH-AM270,Footwear
3,Apple MacBook Pro 14 inch,Laptop with M3 Pro chip 18GB RAM and 512GB SSD...,APL-LT-MBP14,Computers
4,Instant Pot Duo 7-in-1,Electric pressure cooker slow cooker rice cook...,INS-KT-DUO7,Kitchen


In [4]:
def create_chunk(row):
    """Combines product fields into one text block for embedding."""
    return f"Product: {row['name']}. Category: {row['category']}. Description: {row['description']}. SKU: {row['sku']}."
df['chunk'] = df.apply(create_chunk, axis=1)

print("Sample chunk:")
print(df['chunk'].iloc[0])

Sample chunk:
Product: Samsung 55 Inch QLED TV. Category: Electronics. Description: 4K Ultra HD Smart TV with Quantum Dot technology and HDR support. SKU: SAM-TV-55Q.


In [5]:
"""Testing of function call"""

'Testing of function call'

In [6]:
test_row = df.iloc[0]
result = create_chunk(test_row)
print(result)

Product: Samsung 55 Inch QLED TV. Category: Electronics. Description: 4K Ultra HD Smart TV with Quantum Dot technology and HDR support. SKU: SAM-TV-55Q.


In [7]:
# Load a pre-trained embedding model (downloads once, then cached locally)
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Model loaded successfully!")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")

Model loaded successfully!
Embedding dimension: 384


In [8]:
# Convert all product chunks into embeddings
product_embeddings = model.encode(df['chunk'].tolist())

print(f"Shape of embeddings: {product_embeddings.shape}")
print(f"That means: {product_embeddings.shape[0]} products, each with {product_embeddings.shape[1]} numbers")

Shape of embeddings: (12, 384)
That means: 12 products, each with 384 numbers


In [9]:
from numpy import dot
from numpy.linalg import norm

def cosine_similarity(vec1, vec2):
    """Measures how similar two vectors are, from -1 to 1 (1 = identical meaning)."""
    return dot(vec1, vec2) / (norm(vec1) * norm(vec2))

def search_products(query, top_n=3):
    """Find the most relevant products for a given question."""
    
    # Convert the user's question into an embedding
    query_embedding = model.encode(query)
    
    # Compute similarity between the query and EVERY product
    similarities = []
    for i, product_emb in enumerate(product_embeddings):
        score = cosine_similarity(query_embedding, product_emb)
        similarities.append((i, score))
    
    # Sort by similarity score, highest first
    similarities.sort(key=lambda x: x[1], reverse=True)
    
    # Return top N results
    top_results = similarities[:top_n]
    
    print(f"Query: '{query}'\n")
    for idx, score in top_results:
        print(f"Score: {score:.3f} | {df.iloc[idx]['name']}")
        print(f"   {df.iloc[idx]['description']}")
        print()
    
    return top_results

# Test it!
search_products("I need something for noise cancelling")

Query: 'I need something for noise cancelling'

Score: 0.548 | Sony WH-1000XM5 Headphones
   Wireless noise cancelling headphones with 30 hour battery life

Score: 0.279 | Bose SoundLink Flex Speaker
   Waterproof portable Bluetooth speaker with 12 hours of playtime

Score: 0.183 | KitchenAid Stand Mixer 5 Quart
   Tilt-head stand mixer with 10 speeds and stainless steel bowl



[(1, np.float32(0.54784715)),
 (10, np.float32(0.27930617)),
 (9, np.float32(0.18263468))]

In [10]:
search_products("something for cooking rice")

Query: 'something for cooking rice'

Score: 0.393 | Instant Pot Duo 7-in-1
   Electric pressure cooker slow cooker rice cooker steamer and more

Score: 0.259 | KitchenAid Stand Mixer 5 Quart
   Tilt-head stand mixer with 10 speeds and stainless steel bowl

Score: 0.163 | Dell XPS 15 Laptop
   15.6 inch OLED display laptop with Intel Core i7 and 32GB RAM



[(4, np.float32(0.39340666)),
 (9, np.float32(0.25920492)),
 (11, np.float32(0.16285786))]

In [11]:
# Test with a few different queries
search_products("I need something for noise cancelling")
print("=" * 60)
search_products("waterproof speaker")

Query: 'I need something for noise cancelling'

Score: 0.548 | Sony WH-1000XM5 Headphones
   Wireless noise cancelling headphones with 30 hour battery life

Score: 0.279 | Bose SoundLink Flex Speaker
   Waterproof portable Bluetooth speaker with 12 hours of playtime

Score: 0.183 | KitchenAid Stand Mixer 5 Quart
   Tilt-head stand mixer with 10 speeds and stainless steel bowl

Query: 'waterproof speaker'

Score: 0.641 | Bose SoundLink Flex Speaker
   Waterproof portable Bluetooth speaker with 12 hours of playtime

Score: 0.267 | Sony WH-1000XM5 Headphones
   Wireless noise cancelling headphones with 30 hour battery life

Score: 0.116 | Nike Air Max 270 Running Shoes
   Lightweight athletic shoes with Air Max cushioning unit in heel



[(10, np.float32(0.6409086)),
 (1, np.float32(0.266741)),
 (2, np.float32(0.11555384))]

In [13]:
from google import genai
from dotenv import load_dotenv
import os

# Load the API key from .env file
load_dotenv()
gemini_api_key = os.getenv("GEMINI_API_KEY")

client = genai.Client(api_key=gemini_api_key)

# Tiny test call
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Say hello in one short sentence."
)

print(response.text)

d:\mdm_chatbot\venv\lib\site-packages\google\auth\__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
d:\mdm_chatbot\venv\lib\site-packages\google\oauth2\__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)


Hello!


In [16]:
def rag_chatbot(query, top_n=3):
    """Full RAG pipeline: retrieve relevant products, then generate an answer."""
    
    # Retrieve relevant products
    query_embedding = model.encode(query)
    similarities = []
    for i, product_emb in enumerate(product_embeddings):
        score = cosine_similarity(query_embedding, product_emb)
        similarities.append((i, score))
    similarities.sort(key=lambda x: x[1], reverse=True)
    top_results = similarities[:top_n]
    
    # Build context text from retrieved products
    context_parts = []
    for idx, score in top_results:
        product = df.iloc[idx]
        context_parts.append(
            f"- {product['name']} (SKU: {product['sku']}, Category: {product['category']}): {product['description']}"
        )
    context = "\n".join(context_parts)
    
    # Build the prompt for Gemini
    prompt = f"""You are a helpful assistant for a product catalog system.
Answer the user's question using ONLY the product information provided below.
If the answer isn't in the provided products, say you don't have that information.

Respond in a friendly, conversational way, in 2-3 sentences.
Explain WHY the product fits their need, don't just state the product name.

Available products:
{context}

User question: {query}

Answer:"""
    
    # Call Gemini to generate the answer
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    
    return response.text


# Test it!
answer = rag_chatbot("I need something for noise cancelling")
print(answer)

Oh, absolutely! For noise cancelling, I'd highly recommend the Sony WH-1000XM5 Headphones. They're specifically designed with excellent noise cancelling technology to help you find some quiet, wherever you are. Plus, they're wireless and boast an impressive 30-hour battery life!


In [18]:
import time

def call_gemini_with_retry(prompt, max_retries=3, delay=5):
    """Calls Gemini, automatically retrying if the server is temporarily overloaded."""
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=prompt
            )
            return response.text
        except Exception as e:
            if attempt < max_retries - 1:
                print(f"   (Server busy, retrying in {delay}s... attempt {attempt+1}/{max_retries})")
                time.sleep(delay)
            else:
                return f"Sorry, the AI service is temporarily unavailable. Error: {e}"

In [19]:
def rag_chatbot(query, top_n=3):
    """Full RAG pipeline: retrieve relevant products, then generate an answer."""
    
    query_embedding = model.encode(query)
    similarities = []
    for i, product_emb in enumerate(product_embeddings):
        score = cosine_similarity(query_embedding, product_emb)
        similarities.append((i, score))
    similarities.sort(key=lambda x: x[1], reverse=True)
    top_results = similarities[:top_n]
    
    context_parts = []
    for idx, score in top_results:
        product = df.iloc[idx]
        context_parts.append(
            f"- {product['name']} (SKU: {product['sku']}, Category: {product['category']}): {product['description']}"
        )
    context = "\n".join(context_parts)
    
    prompt = f"""You are a helpful assistant for a product catalog system.
Answer the user's question using ONLY the product information provided below.
If the answer isn't in the provided products, say you don't have that information.

Respond in a friendly, conversational way, in 2-3 sentences.
Explain WHY the product fits their need, don't just state the product name.

Available products:
{context}

User question: {query}

Answer:"""
    
    return call_gemini_with_retry(prompt)

In [20]:
test_answer = rag_chatbot("I need something for noise cancelling")
print("Result:", test_answer)

Result: Hi there! For noise cancelling, I'd recommend the Sony WH-1000XM5 Headphones. They are specifically designed with excellent noise cancelling technology, perfect for helping you find some peace and quiet.


In [21]:
questions = [
    "What waterproof products do you have?",
    "I want to bake a cake, what do you recommend?",
    "Do you have any laptops?",
    "What's the cheapest product you have?"
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {rag_chatbot(q)}")
    print("=" * 60)

Q: What waterproof products do you have?
A: We have the Bose SoundLink Flex Speaker, which is a fantastic waterproof portable Bluetooth speaker. This would be a great choice if you're looking for something that can withstand splashes or be used outdoors without worry!
Q: I want to bake a cake, what do you recommend?
A: For baking a cake, I'd recommend the KitchenAid Stand Mixer 5 Quart! Its tilt-head design and 10 speeds are perfect for easily mixing batters and ingredients, making your cake preparation a breeze.
Q: Do you have any laptops?
A: Yes, we certainly do! We have a couple of great laptops available that might interest you.

We have the Dell XPS 15 Laptop, which features a vibrant 15.6-inch OLED display, and the Apple MacBook Pro 14 inch, known for its powerful M3 Pro chip.
Q: What's the cheapest product you have?
A: I'm sorry, but I don't have information about the prices of our products. Therefore, I can't tell you which one is the cheapest at the moment.
